# 05 — The Predictability Horizon

**Can persistent homology forecast where breakthroughs will land?**

We build predictive models using topological features (β₁, persistence entropy)
alongside simple baseline features (cross-class citation count, mixing rate,
patent volume). The key comparison: does topology add information beyond
what simpler metrics provide?

Split: train on pre-2015 breakthroughs, test on post-2015.

---

In [1]:
# %% Imports and setup
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, roc_curve, precision_recall_curve, confusion_matrix, classification_report
from sklearn.preprocessing import StandardScaler

from src.utils import DATA_DIR, get_logger
from src.breakthroughs import load_breakthroughs
from src.metrics import cpc_mixing_rate, shannon_entropy_cpc
from src.plotting import set_style, save_figure, PALETTE, PALETTE_LIST

set_style()
logger = get_logger('nb05')

In [ ]:
# %% Load data
patents = pd.read_parquet(DATA_DIR / 'patents.parquet')
citations = pd.read_parquet(DATA_DIR / 'citations.parquet')
cpc_map = pd.read_parquet(DATA_DIR / 'cpc_map.parquet')
breakthroughs = load_breakthroughs()

# Ensure citing_year column exists
citations['citing_year'] = pd.to_datetime(citations['citing_date']).dt.year

# Load topology results (all 28 pairs, scale-normalized)
from src.topology import compute_all_priority_pairs, ALL_PAIRS
import gc

CACHE_DIR = str(DATA_DIR / 'topology_cache')

pair_results = compute_all_priority_pairs(
    citations, cpc_map,
    cache_dir=CACHE_DIR,
    pairs=ALL_PAIRS,
    start_year=1980,
    end_year=2023,
)

# Build topology_results dict keyed by underscore format for breakthrough lookups
topology_results = {}
for pair, group in pair_results.groupby('section_pair'):
    topology_results[pair] = group.reset_index(drop=True)
    underscore_key = pair.replace('x', '_')
    topology_results[underscore_key] = group.reset_index(drop=True)

print(f'Topology results: {pair_results["section_pair"].nunique()} CPC pairs (of 28)')
gc.collect()

## Build Feature Matrix

For each CPC section pair × year, compute features and label
whether a breakthrough occurred in that pair within the next 5 years.

In [3]:
# %% Build breakthrough lookup: (section_pair, year) → breakthrough occurred within next 5 years
# EXACT pair matching: breakthrough must have its CPC sections match the pair exactly
# This is stricter than "any pair containing any section" — gives a harder, more honest task
bt_lookup = {}

for bt in breakthroughs:
    secs = bt.cpc_sections
    if len(secs) < 2:
        continue
    # Use exact CPC section pair from the breakthrough
    bt_pair = tuple(sorted(secs[:2]))
    pair_key = f'{bt_pair[0]}_{bt_pair[1]}'
    
    # Label years 1-5 before filing (not filing year itself)
    for yr_offset in range(1, 6):
        key = (bt_pair, bt.filing_year - yr_offset)
        bt_lookup[key] = 1

print(f'Unique (pair, year) positive labels: {len(bt_lookup)}')
print(f'Breakthroughs with ≥2 CPC sections: {sum(1 for b in breakthroughs if len(b.cpc_sections) >= 2)}/{len(breakthroughs)}')

Unique (pair, year) positive labels: 70
Breakthroughs with ≥2 CPC sections: 18/34


In [4]:
# %% Assemble feature matrix with genuine simple baselines
# Topology features come from persistent homology
# Simple features: patent volume per CPC pair window + year (no TDA pipeline needed)
rows = []

for pair_key, topo_df in topology_results.items():
    parts = pair_key.split('_')
    if len(parts) != 2:
        continue
    sec_a, sec_b = parts
    pair = tuple(sorted([sec_a, sec_b]))
    
    year_col = 'window_end' if 'window_end' in topo_df.columns else 'year'
    
    for _, row in topo_df.iterrows():
        year = int(row[year_col])
        label = bt_lookup.get((pair, year), 0)
        
        rows.append({
            'section_pair': pair_key,  # For GroupKFold
            'section_a': sec_a,
            'section_b': sec_b,
            'year': year,
            # Topological features (from persistent homology)
            'beta_0': row.get('beta_0', 0),
            'beta_1': row.get('beta_1', 0),
            'beta_2': row.get('beta_2', 0),
            'persistence_entropy': row.get('persistence_entropy', 0),
            'max_persistence_h1': row.get('max_persistence_h1', 0),
            'n_long_lived_h1': row.get('n_long_lived_h1', 0),
            # Simple features (distance matrix summaries — NOT independent of TDA pipeline)
            'n_active_classes': row.get('n_active_classes', 0),
            'mean_distance': row.get('mean_distance', 0),
            'median_distance': row.get('median_distance', 0),
            # Label
            'breakthrough_within_5yr': label,
        })

feat_df = pd.DataFrame(rows)
print(f'Feature matrix: {len(feat_df)} rows')
print(f'Positive labels: {feat_df["breakthrough_within_5yr"].sum()} ({100 * feat_df["breakthrough_within_5yr"].mean():.1f}%)')
print(f'CPC pairs: {feat_df["section_pair"].nunique()}')

Feature matrix: 980 rows
Positive labels: 37 (3.8%)
CPC pairs: 28


In [5]:
# %% Evaluation strategy: Leave-One-Group-Out CV by CPC pair
# Shuffled CV causes massive autocorrelation leakage (adjacent years share 80% data).
# GroupKFold ensures all years for a given CPC pair are in the same fold.
# This tests: can topology in one domain predict breakthroughs in a different domain?
from sklearn.model_selection import LeaveOneGroupOut, GroupKFold

topo_features = ['beta_0', 'beta_1', 'beta_2', 'persistence_entropy', 'max_persistence_h1', 'n_long_lived_h1']
simple_features = ['n_active_classes', 'mean_distance', 'median_distance']
all_features = topo_features + simple_features

X = feat_df[all_features].fillna(0)
y = feat_df['breakthrough_within_5yr']
groups = feat_df['section_pair']

print(f'Full dataset: {len(X)} rows, {int(y.sum())} positive ({100*y.mean():.1f}%)')
print(f'Groups (CPC pairs): {groups.nunique()}')
print(f'Using Leave-One-Group-Out CV (each fold holds out one CPC pair)')
print(f'\nCAVEAT: "simple" features (mean/median distance) derive from the same')
print(f'co-citation matrix as topology features. They are not truly independent baselines.')

Full dataset: 980 rows, 37 positive (3.8%)
Groups (CPC pairs): 28
Using Leave-One-Group-Out CV (each fold holds out one CPC pair)

CAVEAT: "simple" features (mean/median distance) derive from the same
co-citation matrix as topology features. They are not truly independent baselines.


## Train Models

Three variants:
1. Topology-only features
2. Simple-only features (patent/citation volume)
3. Combined

In [6]:
# %% Train and evaluate models with Leave-One-Group-Out CV
# Each fold holds out one CPC pair entirely — no autocorrelation leakage
results = {}
logo = LeaveOneGroupOut()

feature_sets = {
    'Topology only': topo_features,
    'Simple only': simple_features,
    'Combined': all_features,
}

models = {
    'Logistic Regression': lambda: LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
    'Random Forest': lambda: RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42),
}

for model_name, model_fn in models.items():
    for feat_name, feat_cols in feature_sets.items():
        key = f'{model_name} — {feat_name}'
        
        X_subset = X[feat_cols].values
        
        # Collect out-of-fold predictions
        y_probs = np.zeros(len(y))
        y_preds = np.zeros(len(y), dtype=int)
        
        fold_aucs = []
        for train_idx, test_idx in logo.split(X_subset, y, groups):
            # Fit scaler INSIDE the loop (no leakage)
            scaler = StandardScaler()
            X_train = scaler.fit_transform(X_subset[train_idx])
            X_test = scaler.transform(X_subset[test_idx])
            
            model = model_fn()
            model.fit(X_train, y.iloc[train_idx])
            
            if hasattr(model, 'predict_proba'):
                y_probs[test_idx] = model.predict_proba(X_test)[:, 1]
            else:
                y_probs[test_idx] = model.decision_function(X_test)
            
            y_preds[test_idx] = model.predict(X_test)
            
            # Some folds may have no positives — skip AUC for those
            if y.iloc[test_idx].sum() > 0 and y.iloc[test_idx].sum() < len(test_idx):
                try:
                    fold_aucs.append(roc_auc_score(y.iloc[test_idx], y_probs[test_idx]))
                except ValueError:
                    pass
        
        try:
            overall_auc = roc_auc_score(y, y_probs)
        except ValueError:
            overall_auc = 0.5
        
        # Train final model on all data for feature importance
        scaler_final = StandardScaler()
        X_all_scaled = scaler_final.fit_transform(X_subset)
        final_model = model_fn()
        final_model.fit(X_all_scaled, y)
        
        results[key] = {
            'model': final_model,
            'y_prob': y_probs,
            'y_pred': y_preds,
            'auc': overall_auc,
            'fold_aucs': fold_aucs,
            'features': feat_cols,
        }
        
        fold_str = ''
        if fold_aucs:
            fold_str = f' (valid folds: {len(fold_aucs)}/{groups.nunique()}, mean={np.mean(fold_aucs):.3f})'
        print(f'{key}: AUC = {overall_auc:.3f}{fold_str}')

Logistic Regression — Topology only: AUC = 0.609 (valid folds: 3/28, mean=0.528)
Logistic Regression — Simple only: AUC = 0.270 (valid folds: 3/28, mean=0.459)


Logistic Regression — Combined: AUC = 0.601 (valid folds: 3/28, mean=0.600)


Random Forest — Topology only: AUC = 0.394 (valid folds: 3/28, mean=0.499)


Random Forest — Simple only: AUC = 0.411 (valid folds: 3/28, mean=0.568)


Random Forest — Combined: AUC = 0.373 (valid folds: 3/28, mean=0.500)


## Figure 1: ROC Curves

In [7]:
# %% Figure 1: ROC curves (Leave-One-Group-Out, out-of-fold predictions)
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for idx, model_name in enumerate(['Logistic Regression', 'Random Forest']):
    ax = axes[idx]
    
    for i, (feat_name, _) in enumerate(feature_sets.items()):
        key = f'{model_name} — {feat_name}'
        if key in results:
            fpr, tpr, _ = roc_curve(y, results[key]['y_prob'])
            ax.plot(fpr, tpr, color=PALETTE_LIST[i], linewidth=2,
                    label=f"{feat_name} (AUC={results[key]['auc']:.3f})")
    
    ax.plot([0, 1], [0, 1], 'k--', alpha=0.3)
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.set_title(model_name)
    ax.legend(loc='lower right')
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)

fig.suptitle('Figure 1: ROC Curves — Leave-One-Group-Out CV by CPC Pair', fontsize=14, y=1.02)
fig.tight_layout()
save_figure(fig, '05_roc_curves')

2026-03-20 21:32:52 | src.plotting             | INFO    | Saved figure: /Users/christopherortiz/Desktop/the-shape-of-discovery/figures/05_roc_curves.png


PosixPath('/Users/christopherortiz/Desktop/the-shape-of-discovery/figures/05_roc_curves.png')

## Figure 2: Feature Importance (SHAP)

In [8]:
# %% Figure 2: Feature importance for the combined Random Forest
rf_combined_key = 'Random Forest — Combined'
if rf_combined_key in results:
    importances = results[rf_combined_key]['model'].feature_importances_
    sorted_idx = np.argsort(importances)[::-1]
    
    fig, ax = plt.subplots(figsize=(10, 6))
    colors = [PALETTE['red'] if all_features[i] in topo_features else PALETTE['blue'] for i in sorted_idx]
    ax.barh(range(len(all_features)), importances[sorted_idx], color=colors)
    ax.set_yticks(range(len(all_features)))
    ax.set_yticklabels([all_features[i] for i in sorted_idx])
    ax.set_xlabel('Feature Importance (Gini)')
    ax.set_title('Figure 2: RF Feature Importance (Red = Topology, Blue = Simple)')
    ax.invert_yaxis()
    fig.tight_layout()
    save_figure(fig, '05_feature_importance')

2026-03-20 21:32:52 | src.plotting             | INFO    | Saved figure: /Users/christopherortiz/Desktop/the-shape-of-discovery/figures/05_feature_importance.png


## Figure 3: Confusion Matrix

In [9]:
# %% Figure 3: Confusion matrix for best model (LOGO out-of-fold predictions)
best_key = max(results.keys(), key=lambda k: results[k]['auc'])
best = results[best_key]

cm = confusion_matrix(y, best['y_pred'])

fig, ax = plt.subplots(figsize=(8, 6))
import seaborn as sns
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['No Breakthrough', 'Breakthrough'],
            yticklabels=['No Breakthrough', 'Breakthrough'])
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title(f'Figure 3: Confusion Matrix — {best_key}\n(Leave-One-Group-Out CV, AUC={best["auc"]:.3f})')
fig.tight_layout()
save_figure(fig, '05_confusion_matrix')

print(f'\nBest model: {best_key}')
print(f'AUC: {best["auc"]:.3f}')
print(f'\n{classification_report(y, best["y_pred"], target_names=["No BT", "BT"])}')

# Naive baseline comparison
naive_acc = 1 - y.mean()
print(f'Naive "always predict no breakthrough" accuracy: {naive_acc:.3f}')

2026-03-20 21:32:52 | src.plotting             | INFO    | Saved figure: /Users/christopherortiz/Desktop/the-shape-of-discovery/figures/05_confusion_matrix.png



Best model: Logistic Regression — Topology only
AUC: 0.609

              precision    recall  f1-score   support

       No BT       0.97      0.66      0.78       943
          BT       0.06      0.57      0.11        37

    accuracy                           0.65       980
   macro avg       0.52      0.61      0.45       980
weighted avg       0.94      0.65      0.76       980

Naive "always predict no breakthrough" accuracy: 0.962


# %% Interpretation — honest assessment
print('=== Honest Assessment ===')
print()

# Find best topology-only and simple-only AUCs
topo_lr = results.get('Logistic Regression — Topology only', {}).get('auc', 0)
simple_lr = results.get('Logistic Regression — Simple only', {}).get('auc', 0)
topo_rf = results.get('Random Forest — Topology only', {}).get('auc', 0)
simple_rf = results.get('Random Forest — Simple only', {}).get('auc', 0)
combined_lr = results.get('Logistic Regression — Combined', {}).get('auc', 0)
combined_rf = results.get('Random Forest — Combined', {}).get('auc', 0)

print(f'Logistic Regression:  Topo={topo_lr:.3f}  Simple={simple_lr:.3f}  Combined={combined_lr:.3f}')
print(f'Random Forest:        Topo={topo_rf:.3f}  Simple={simple_rf:.3f}  Combined={combined_rf:.3f}')

print()
print('Key findings:')
if topo_lr > simple_lr + 0.05:
    print(f'  - LR: Topology ({topo_lr:.3f}) outperforms simple ({simple_lr:.3f}) by {topo_lr-simple_lr:.3f}')
elif simple_lr > topo_lr + 0.05:
    print(f'  - LR: Simple ({simple_lr:.3f}) outperforms topology ({topo_lr:.3f}) by {simple_lr-topo_lr:.3f}')
else:
    print(f'  - LR: Topology ({topo_lr:.3f}) and simple ({simple_lr:.3f}) perform similarly')

if topo_rf > simple_rf + 0.05:
    print(f'  - RF: Topology ({topo_rf:.3f}) outperforms simple ({simple_rf:.3f}) by {topo_rf-simple_rf:.3f}')
elif simple_rf > topo_rf + 0.05:
    print(f'  - RF: Simple ({simple_rf:.3f}) outperforms topology ({topo_rf:.3f}) by {simple_rf-topo_rf:.3f}')
else:
    print(f'  - RF: Topology ({topo_rf:.3f}) and simple ({simple_rf:.3f}) perform similarly')

print()
print('Caveats (critical for honest interpretation):')
print('  1. Leave-One-Group-Out by CPC pair tests cross-domain generalization,')
print('     NOT temporal forecasting. AUCs reflect whether topology patterns')
print('     transfer between technology domains, not whether they predict the future.')
print('  2. "Simple" features (mean/median distance) come from the same co-citation')
print('     matrix as topology features. They are not truly independent baselines.')
print('  3. Dataset is small (350 rows, 10 groups). Results are noisy.')
print('  4. Multiple breakthroughs share CPC pairs, inflating positive labels.')
print()

all_near_chance = all(v < 0.6 for v in [topo_lr, simple_lr, topo_rf, simple_rf])
if all_near_chance:
    print('CONCLUSION: All models near chance level. The dataset is too small for')
    print('reliable predictive modeling. Topological features do not demonstrably')
    print('forecast breakthroughs across technology domains.')
else:
    print('CONCLUSION: Some predictive signal exists, but interpretation requires')
    print('caution given the small dataset and shared feature pipeline. The most')
    print('reliable comparison is Logistic Regression (scale-sensitive, so the')
    print('topology-vs-simple gap is meaningful if present).')